In [ ]:
# %%
from Det import Det, DetData
import numpy as np
import matplotlib.pyplot as plt
from rich.table import Table
from rich import print
from hdf5storage import loadmat
import time
from AcqFunc import *

# %%
srv = DetData("10.20.99.2", port=50099)
dets = srv.findDet()
det = list(dets.items())[0][1]
# det = Det("10.20.22.240")
# det.addQueue(srv.addDet("10.20.22.240"))
srv.listen()

# %% 状态信息
def dictPrint(d: dict, title="", k="", v=""):
    t = Table(title=title)
    t.add_column(k)
    t.add_column(v)
    [t.add_row(k, str(v)) for k, v in d.items()]
    print(t)
dictPrint(det.statusTemperature(), "温度")
dictPrint(det.statusPosition(0.0375), "位置")
dictPrint(det.statusPower(), "电源")
dictPrint(det.statusPowerSwitch(), "开关")
dictPrint(det.statusFanSpeed(), "风扇")

# %% 电源及参数设置
det.setPositionConfig([
    {"pos": 0, "en": 1, "polarity": 0, "clearPos": 1, "zeroShift": 0},
    {"pos": 1, "en": 0, "polarity": 0, "clearPos": 1, "zeroShift": 0}])
det.setPowerSwitch({
    "laser1": 0, "laser0": 0, "opa": 1, "vbias": 1, "vcc12": 1, "vdd25": 1})
det.DetectRegSet(0x0018, 0x600003FF)
det.setWinNum(4)

# %%
for i in range(1):
    # tube = f""
    tube = f"_70kV_10mA"
    filter = f"_0.3mmCu+0.3mmSn"
    speed = f"_150mmps"
    name = f"Acq260109/limit_exp{speed}{filter}{tube}"

    subname = f"{name}"
    det.setWinRange(0, 0, 99)
    # time.sleep(4)
    print("detector run")
    # time.sleep(1)
    data = histAcqNoMove(det, cnt=None, time=4, interval = int(4 * 10))
    saveHist(data, subname, None)
    print(subname)
    showHist(
        data,
        pos_en=True,
        pos_step=0.0375,
        cal_sel=(400, 450),
        rate=1400/20,
        log_en=False,
        pos_limit=(0, 0),
        caxis=(0, 0),
        save_png=""
    )

# %%
for i in range(1):
    # tube = f""
    tube = f"_75kV_110mA"
    # filter = f"_0.3mmCu+0.3mmSn"
    filter = f"_25mmAl"
    speed = f"_37.5mmps"
    freq = 500
    name = f"Acq260109/fblk_deg45{speed}{filter}{tube}_{freq}hz"

    subname = f"{name}"
    det.setWinRange(0, 0, 99)
    det.setWinRange(1, 6, 21)
    det.setWinRange(2, 22, 40)
    det.setWinRange(3, 0, 90)
    print("detector run")
    data = thrAcqNoMove(det, cnt=None, time=8, interval = int(1000/freq * 10))
    saveThr(data, subname)
    print(subname)
    data2 = {
        "pos0h": data["pos0h"],
        "data": data["data"].reshape((data["data"].shape[0], -1, data["data"].shape[3]))
    }
    showHist(
        data2,
        pos_en=True,
        pos_step=0.0375,
        cal_sel=(400, 450),
        rate=1400/20,
        log_en=False,
        pos_limit=(0, 0),
        caxis=(0, 0),
        save_png=""
    )

# %%
# data_save = loadmat(
#     r"C:\Users\RJYL_Hardware_Test1\Documents\VPlusDet\Acq1216\gujia_200mmps_0.3mmCu_110kV_50mA.mat"
# )
# data = {
#     "data": np.transpose(data_save["d"]["data"], (2, 1, 0)),
#     "pos0h": data_save["d"]["pos"][:, None]
# }
# data["data"] = data["data"][:, :, :]
plt.figure()
plt.plot(data["data"].sum(axis=0).T)
plt.show()
plt.figure()
plt.plot(data["data"].sum(axis=0).sum(axis=1).T)
plt.show()
showHist(
    data,
    pos_en=True,
    pos_step=0.0375,
    cal_sel=(0, 0),
    rate=1400/500,
    log_en=False,
    pos_limit=(0, 0),
    caxis=(0, 0),
    save_png=""
)
plt.figure()
plt.plot(data["data"][:,300:320,:].sum(axis=2))
plt.ylim([0,400])

# %%
